# release16_checkpoint_distillation_bc_repair_screen_v1

Notebook-first model-selection diagnostic for the release16 checkpoint-distillation BC handoff. The previous readiness screen failed only one required sparse slice: `wf_2023_2024` `release_gate_state` accuracy was 36.36% with all release-gate rows predicted as action 4. This run screens small BC sample-weight recipes before any PPO run.

No actor checkpoint is retained or promoted from this notebook.


In [1]:

from __future__ import annotations

import json
import math
import random
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "SPEC.md").exists() and (candidate / "src/soxl_quant").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    state_objective_bc_sample_weights,
)
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

EXPERIMENT = "release16_checkpoint_distillation_bc_repair_screen_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
SOURCE_CONTRACT_CARD = ROOT / "runtime/model_selection/card/release16_checkpoint_distillation_ytd_alignment_contract_v1"
SOURCE_READINESS_CARD = ROOT / "runtime/model_selection/card/release16_checkpoint_distillation_bc_readiness_v1"
RETENTION_GATE = ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json"
TARGET_CONTRACT = SOURCE_CONTRACT_CARD / "distillation_target_contract.csv"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
SEED = 20260615

CARD.mkdir(parents=True, exist_ok=True)

WALK_FORWARD_WINDOWS = {
    "wf_2020": {
        "target_fold": "wf_2020_train",
        "train": ("2010-03-11", "2018-12-31"),
        "valid": ("2019-01-01", "2019-12-31"),
        "test": ("2020-01-01", "2020-12-31"),
    },
    "wf_2022": {
        "target_fold": "wf_2022_train",
        "train": ("2010-03-11", "2020-12-31"),
        "valid": ("2021-01-01", "2021-12-31"),
        "test": ("2022-01-01", "2022-12-31"),
    },
    "wf_2023_2024": {
        "target_fold": "wf_2023_2024_train",
        "train": ("2010-03-11", "2022-12-31"),
        "valid": ("2023-01-01", "2023-12-31"),
        "test": ("2024-01-01", "2024-12-31"),
    },
}

SLICE_RULES = {
    "all": {"min_samples": 100, "min_accuracy": 0.55, "required": True},
    "release_gate_state": {"min_samples": 5, "min_accuracy": 0.60, "required": True},
    "is_rebound_cluster": {"min_samples": 5, "min_accuracy": 0.55, "required": False},
    "is_downtrend_state": {"min_samples": 100, "min_accuracy": 0.65, "required": False},
    "drawdown_control_state": {"min_samples": 20, "min_accuracy": 0.55, "required": False},
}
SLICE_COLS = tuple(k for k in SLICE_RULES if k != "all")

CONFIG_GRID = [
    {
        "config": "current_readiness_baseline",
        "steps": 400,
        "lr": 0.001,
        "batch_size": 128,
        "rebound_multiplier": 6.0,
        "downtrend_multiplier": 2.0,
        "release_gate_multiplier": 16.0,
        "ordinary_multiplier": 1.0,
        "objective_weight_power": 0.20,
        "class_balanced_beta": 0.99,
        "weight_floor": 0.05,
        "weight_cap": 35.0,
        "normalize_mean": True,
    },
    {
        "config": "release64_cb99_obj020",
        "steps": 500,
        "lr": 0.001,
        "batch_size": 128,
        "rebound_multiplier": 6.0,
        "downtrend_multiplier": 2.0,
        "release_gate_multiplier": 64.0,
        "ordinary_multiplier": 1.0,
        "objective_weight_power": 0.20,
        "class_balanced_beta": 0.99,
        "weight_floor": 0.05,
        "weight_cap": 80.0,
        "normalize_mean": True,
    },
    {
        "config": "release128_cb995_obj020",
        "steps": 500,
        "lr": 0.001,
        "batch_size": 128,
        "rebound_multiplier": 6.0,
        "downtrend_multiplier": 2.0,
        "release_gate_multiplier": 128.0,
        "ordinary_multiplier": 1.0,
        "objective_weight_power": 0.20,
        "class_balanced_beta": 0.995,
        "weight_floor": 0.05,
        "weight_cap": 140.0,
        "normalize_mean": True,
    },
    {
        "config": "release128_cb995_obj000",
        "steps": 500,
        "lr": 0.001,
        "batch_size": 128,
        "rebound_multiplier": 6.0,
        "downtrend_multiplier": 2.0,
        "release_gate_multiplier": 128.0,
        "ordinary_multiplier": 1.0,
        "objective_weight_power": 0.0,
        "class_balanced_beta": 0.995,
        "weight_floor": 0.05,
        "weight_cap": 140.0,
        "normalize_mean": True,
    },
    {
        "config": "release256_cb995_obj020",
        "steps": 600,
        "lr": 0.001,
        "batch_size": 128,
        "rebound_multiplier": 6.0,
        "downtrend_multiplier": 2.0,
        "release_gate_multiplier": 256.0,
        "ordinary_multiplier": 1.0,
        "objective_weight_power": 0.20,
        "class_balanced_beta": 0.995,
        "weight_floor": 0.05,
        "weight_cap": 240.0,
        "normalize_mean": True,
    },
    {
        "config": "release256_beta0_obj020",
        "steps": 600,
        "lr": 0.001,
        "batch_size": 128,
        "rebound_multiplier": 6.0,
        "downtrend_multiplier": 2.0,
        "release_gate_multiplier": 256.0,
        "ordinary_multiplier": 1.0,
        "objective_weight_power": 0.20,
        "class_balanced_beta": 0.0,
        "weight_floor": 0.05,
        "weight_cap": 240.0,
        "normalize_mean": True,
    },
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == object:
        return series.fillna(False).astype(str).str.lower().isin({"true", "1", "yes", "y", "t"})
    return series.fillna(False).astype(bool)


def frame_date_bounds(frame: pd.DataFrame) -> tuple[str | None, str | None]:
    idx = frame.index
    if isinstance(idx, pd.MultiIndex):
        level = "datetime" if "datetime" in idx.names else 0
        values = idx.get_level_values(level)
    else:
        values = idx
    dates = pd.to_datetime(values).normalize()
    if len(dates) == 0:
        return None, None
    return str(dates.min().date()), str(dates.max().date())


def build_replay_feature_frame(profile):
    dataset = build_dataset(
        train_seg=("2010-03-11", "2022-12-31"),
        valid_seg=("2023-01-01", "2023-12-31"),
        test_seg=("2024-01-01", "2024-12-31"),
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    parts = [segment_frame(dataset, name) for name in ("train", "valid", "test")]
    frame = pd.concat(parts).sort_index()
    return frame.loc[~frame.index.duplicated(keep="last")].copy()


def slice_gate_from_metrics(slice_metrics: pd.DataFrame) -> tuple[bool, dict[str, Any]]:
    details: dict[str, Any] = {}
    failed: list[str] = []
    indexed = slice_metrics.set_index("slice") if not slice_metrics.empty else pd.DataFrame()
    for name, rule in SLICE_RULES.items():
        if name not in indexed.index:
            row = {"samples": 0, "accuracy": None, "loss": None, "gate": False, **rule}
        else:
            metric = indexed.loc[name]
            samples = int(metric.get("samples", 0))
            accuracy = float(metric.get("accuracy", float("nan")))
            loss = float(metric.get("loss", float("nan")))
            enough_samples = samples >= int(rule["min_samples"])
            enough_accuracy = math.isfinite(accuracy) and accuracy >= float(rule["min_accuracy"])
            row = {
                "samples": samples,
                "accuracy": accuracy,
                "loss": loss,
                "gate": bool(enough_samples and enough_accuracy),
                **rule,
            }
        details[name] = row
        if bool(rule["required"]) and not bool(row["gate"]):
            failed.append(name)
    return len(failed) == 0, {"failed_required_slices": failed, "slice_details": details}


def training_config(row: dict[str, Any], *, seed: int) -> StateObjectiveBCTrainingConfig:
    return StateObjectiveBCTrainingConfig(
        steps=int(row["steps"]),
        lr=float(row["lr"]),
        batch_size=int(row["batch_size"]),
        seed=int(seed),
    )


def weight_config(row: dict[str, Any]) -> StateObjectiveBCSampleWeightConfig:
    return StateObjectiveBCSampleWeightConfig(
        rebound_multiplier=float(row["rebound_multiplier"]),
        downtrend_multiplier=float(row["downtrend_multiplier"]),
        release_gate_multiplier=float(row["release_gate_multiplier"]),
        ordinary_multiplier=float(row["ordinary_multiplier"]),
        objective_weight_power=float(row["objective_weight_power"]),
        class_balanced_beta=float(row["class_balanced_beta"]),
        weight_floor=float(row["weight_floor"]),
        weight_cap=float(row["weight_cap"]),
        normalize_mean=bool(row["normalize_mean"]),
    )


def actor_predictions_and_losses(actor: Any, obs_array: np.ndarray, action_array: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if len(obs_array) == 0:
        return np.asarray([], dtype=np.int64), np.asarray([], dtype=np.float32)
    device = next(actor.parameters()).device
    obs_tensor = torch.as_tensor(obs_array, dtype=torch.float32, device=device)
    action_tensor = torch.as_tensor(action_array, dtype=torch.long, device=device)
    loss_fn = torch.nn.CrossEntropyLoss(reduction="none")
    with torch.no_grad():
        logits = actor(obs_tensor)
        if isinstance(logits, tuple):
            logits = logits[0]
        predictions = logits.argmax(-1).detach().cpu().numpy().astype(np.int64)
        losses = loss_fn(logits, action_tensor).detach().cpu().numpy().astype(np.float32)
    return predictions, losses


def release_gate_trace(actor: Any, obs: np.ndarray, actions: np.ndarray, metadata: pd.DataFrame) -> pd.DataFrame:
    predictions, losses = actor_predictions_and_losses(actor, obs, actions)
    work = metadata.copy().reset_index(drop=True)
    work["label_action"] = np.asarray(actions, dtype=np.int64)
    work["predicted_action"] = predictions
    work["prediction_loss"] = losses
    if "release_gate_state" not in work.columns:
        return pd.DataFrame()
    trace = work[work["release_gate_state"].fillna(False).astype(bool)].copy()
    keep_cols = [
        "datetime",
        "label_action",
        "predicted_action",
        "prediction_loss",
        "teacher_position",
        "rebound_target_position",
        "risk_target_position",
        "state_objective_target_position",
        "objective_weight",
        "is_rebound_cluster",
        "is_downtrend_state",
        "release_gate_state",
        "drawdown_control_state",
    ]
    keep_cols = [col for col in keep_cols if col in trace.columns]
    return trace[keep_cols].copy()


def candidate_sort_key(frame: pd.DataFrame) -> pd.DataFrame:
    sort_cols = [
        "all_required_fit_gate",
        "release_gate_pass_count",
        "wf_2023_2024_release_gate_accuracy",
        "release_gate_accuracy_min",
        "all_accuracy_min",
        "after_accuracy_mean",
        "after_loss_mean",
    ]
    work = frame.copy()
    for col in sort_cols:
        if col not in work.columns:
            work[col] = np.nan
    return work.sort_values(
        [
            "all_required_fit_gate",
            "release_gate_pass_count",
            "wf_2023_2024_release_gate_accuracy",
            "release_gate_accuracy_min",
            "all_accuracy_min",
            "after_accuracy_mean",
            "after_loss_mean",
        ],
        ascending=[False, False, False, False, False, False, True],
        na_position="last",
    )

print("root", ROOT)
print("card", CARD)


root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_checkpoint_distillation_bc_repair_screen_v1


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Governance

- Baselines: canonical `soxlbacktest_policy_reimplementation_v1`; direct comparison `release16_checkpoint_distillation_bc_readiness_v1`.
- Role: `diagnose` BC handoff repair before PPO.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Changed layer: `model_selection` / `bc_sample_weighting`; no execution-layer or dashboard source changes.
- Hypothesis: the existing target contract can proceed only if a small sample-weight repair makes all required BC slices pass without losing global BC fit.
- Acceptance: all required folds evaluated, target scope non-leaking, feature coverage OK, all required slice gates pass, and no actor is retained/promoted.


In [2]:

profile = load_profile(PROFILE_NAME)
targets = pd.read_csv(TARGET_CONTRACT)
targets["datetime"] = pd.to_datetime(targets["datetime"]).dt.normalize()
for col in (
    "training_label_allowed",
    "is_rebound_cluster",
    "is_downtrend_state",
    "release_gate_state",
    "drawdown_control_state",
):
    if col in targets.columns:
        targets[col] = normalize_bool_series(targets[col])

source_contract_summary = load_json(SOURCE_CONTRACT_CARD / "summary.json")
source_readiness_summary = load_json(SOURCE_READINESS_CARD / "summary.json")
retention_gate = load_json(RETENTION_GATE)
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
action_levels = list(action_levels_for_profile(profile))

scope_audit = {
    "rows": int(len(targets)),
    "training_label_allowed_rows": int(targets.get("training_label_allowed", pd.Series(False, index=targets.index)).sum()),
    "blocked_training_label_rows": int((~targets.get("training_label_allowed", pd.Series(False, index=targets.index))).sum()),
    "target_scope_values": sorted(targets.get("target_scope", pd.Series(dtype=object)).dropna().astype(str).unique().tolist()),
    "label_role_values": sorted(targets.get("label_role", pd.Series(dtype=object)).dropna().astype(str).unique().tolist()),
    "oos_guard_values": sorted(targets.get("oos_or_ytd_direct_label_guard", pd.Series(dtype=object)).dropna().astype(str).unique().tolist()),
    "min_date": str(targets["datetime"].min().date()),
    "max_date": str(targets["datetime"].max().date()),
}
scope_audit["target_scope_gate"] = (
    scope_audit["blocked_training_label_rows"] == 0
    and scope_audit["target_scope_values"] == ["fold_local"]
    and scope_audit["oos_guard_values"] == ["allowed_train_dates_only"]
)

config_grid = pd.DataFrame(CONFIG_GRID)
config_grid.to_csv(CARD / "config_grid.csv", index=False)
pd.DataFrame([scope_audit]).to_csv(CARD / "target_scope_audit.csv", index=False)

print("profile", profile.name, "feature_count", len(feature_columns), "action_levels", action_levels)
print("source_readiness_decision", source_readiness_summary.get("decision"))
print("scope_audit", scope_audit)
display(config_grid)


profile sparse_release_bc_ppo_screen_v1 feature_count 59 action_levels [np.float32(0.0), np.float32(0.1), np.float32(0.2), np.float32(0.3), np.float32(0.4), np.float32(0.5), np.float32(0.6), np.float32(0.7), np.float32(0.8), np.float32(0.9), np.float32(1.0)]
source_readiness_decision bc_contract_readiness_failed_repair_before_training
scope_audit {'rows': 3731, 'training_label_allowed_rows': 3731, 'blocked_training_label_rows': 0, 'target_scope_values': ['fold_local'], 'label_role_values': ['fold_local_train_label'], 'oos_guard_values': ['allowed_train_dates_only'], 'min_date': '2010-03-11', 'max_date': '2022-12-30', 'target_scope_gate': True}


,config,steps,lr,batch_size,rebound_multiplier,downtrend_multiplier,release_gate_multiplier,ordinary_multiplier,objective_weight_power,class_balanced_beta,weight_floor,weight_cap,normalize_mean
0,current_readiness_baseline,400,0.001,128,6.0,2.0,16.0,1.0,0.2,0.990,0.05,35.0,True
1,release64_cb99_obj020,500,0.001,128,6.0,2.0,64.0,1.0,0.2,0.990,0.05,80.0,True
2,release128_cb995_obj020,500,0.001,128,6.0,2.0,128.0,1.0,0.2,0.995,0.05,140.0,True
3,release128_cb995_obj000,500,0.001,128,6.0,2.0,128.0,1.0,0.0,0.995,0.05,140.0,True
4,release256_cb995_obj020,600,0.001,128,6.0,2.0,256.0,1.0,0.2,0.995,0.05,240.0,True
5,release256_beta0_obj020,600,0.001,128,6.0,2.0,256.0,1.0,0.2,0.000,0.05,240.0,True


In [3]:

init_qlib(QLIB_DATA_PATH)
frame = build_replay_feature_frame(profile)
frame_min, frame_max = frame_date_bounds(frame)
feature_schema = pd.DataFrame({"feature": feature_columns})
feature_schema.to_csv(CARD / "feature_schema.csv", index=False)
feature_summary = {
    "profile": profile.name,
    "feature_packs": list(profile.feature.packs),
    "feature_count": int(len(feature_columns)),
    "feature_frame_rows": int(len(frame)),
    "feature_frame_columns": int(len(frame.columns)),
    "feature_frame_min_date": frame_min,
    "feature_frame_max_date": frame_max,
}
write_json(CARD / "feature_summary.json", feature_summary)
print(feature_summary)


[70521:MainThread](2026-06-15 05:40:40,615) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[70521:MainThread](2026-06-15 05:40:41,078) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[70521:MainThread](2026-06-15 05:40:41,078) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[70521:MainThread](2026-06-15 05:40:42,298) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[70521:MainThread](2026-06-15 05:40:42,298) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[70521:MainThread](2026-06-15 05:40:42,298) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


{'profile': 'sparse_release_bc_ppo_screen_v1', 'feature_packs': [], 'feature_count': 59, 'feature_frame_rows': 3475, 'feature_frame_columns': 59, 'feature_frame_min_date': '2011-03-11', 'feature_frame_max_date': '2024-12-31'}


In [4]:

fold_targets_by_fold: dict[str, pd.DataFrame] = {}
fold_dataset_by_fold: dict[str, Any] = {}
dataset_rows: list[dict[str, Any]] = []

for fold, spec in WALK_FORWARD_WINDOWS.items():
    train_start = pd.Timestamp(spec["train"][0])
    train_end = pd.Timestamp(spec["train"][1])
    valid_start = pd.Timestamp(spec["valid"][0])
    valid_end = pd.Timestamp(spec["valid"][1])
    test_start = pd.Timestamp(spec["test"][0])
    test_end = pd.Timestamp(spec["test"][1])

    fold_all = targets[targets["fold"].astype(str).eq(spec["target_fold"])].copy()
    fold_targets = fold_all[fold_all["datetime"].between(train_start, train_end)].copy()
    valid_overlap = fold_all[fold_all["datetime"].between(valid_start, valid_end)]
    test_overlap = fold_all[fold_all["datetime"].between(test_start, test_end)]
    label_allowed = fold_targets.get("training_label_allowed", pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool)
    fold_targets = fold_targets[label_allowed].copy()

    target_min = str(fold_targets["datetime"].min().date()) if not fold_targets.empty else None
    target_max = str(fold_targets["datetime"].max().date()) if not fold_targets.empty else None
    row = {
        "fold": fold,
        "target_fold": spec["target_fold"],
        "target_rows": int(len(fold_targets)),
        "valid_overlap_rows": int(len(valid_overlap)),
        "test_overlap_rows": int(len(test_overlap)),
        "target_min_date": target_min,
        "target_max_date": target_max,
        "target_scope_ok": bool(len(fold_targets) > 0 and len(test_overlap) == 0),
    }
    if not fold_targets.empty:
        dataset = build_state_objective_bc_dataset_from_frame(
            frame,
            targets=fold_targets,
            profile=profile,
            sample_size=0,
            rng_seed=SEED,
        )
        fold_targets_by_fold[fold] = fold_targets
        fold_dataset_by_fold[fold] = dataset
        row.update(
            {
                "bc_samples": int(len(dataset.actions)),
                "metadata_rows": int(len(dataset.metadata)),
                "unique_label_actions": int(pd.Series(dataset.actions).nunique()),
                "release_gate_samples": int(dataset.metadata.get("release_gate_state", pd.Series(False, index=dataset.metadata.index)).fillna(False).astype(bool).sum()),
                "feature_coverage_ok": bool(len(dataset.actions) > 0),
            }
        )
    dataset_rows.append(row)

dataset_summary = pd.DataFrame(dataset_rows)
dataset_summary.to_csv(CARD / "dataset_summary.csv", index=False)
display(dataset_summary)


,fold,target_fold,target_rows,valid_overlap_rows,test_overlap_rows,target_min_date,target_max_date,target_scope_ok,bc_samples,metadata_rows,unique_label_actions,release_gate_samples,feature_coverage_ok
0,wf_2020,wf_2020_train,2218,252,0,2010-03-11,2018-12-31,True,1905,1905,5,32,True
1,wf_2022,wf_2022_train,253,252,0,2020-01-02,2020-12-31,True,253,253,5,8,True
2,wf_2023_2024,wf_2023_2024_train,756,0,0,2020-01-02,2022-12-30,True,756,756,5,11,True


In [5]:

fold_rows: list[dict[str, Any]] = []
slice_frames: list[pd.DataFrame] = []
weight_rows: list[dict[str, Any]] = []
trace_frames: list[pd.DataFrame] = []
action_frames: list[pd.DataFrame] = []

for config_index, raw_config in enumerate(CONFIG_GRID):
    config_name = str(raw_config["config"])
    print("config", config_name)
    train_cfg = training_config(raw_config, seed=SEED + config_index)
    weight_cfg = weight_config(raw_config)

    for fold, spec in WALK_FORWARD_WINDOWS.items():
        print("  fold", fold)
        fold_targets = fold_targets_by_fold.get(fold)
        dataset = fold_dataset_by_fold.get(fold)
        if fold_targets is None or dataset is None:
            fold_rows.append({"config": config_name, "fold": fold, "status": "blocked_no_fold_targets"})
            continue

        set_seed(SEED + config_index)
        weights = state_objective_bc_sample_weights(dataset.metadata, actions=dataset.actions, config=weight_cfg)
        release_mask = dataset.metadata.get("release_gate_state", pd.Series(False, index=dataset.metadata.index)).fillna(False).astype(bool).to_numpy()
        release_weight_sum = float(weights[release_mask].sum()) if release_mask.any() else 0.0
        weight_sum = float(weights.sum()) if len(weights) else 0.0
        weight_rows.append(
            {
                "config": config_name,
                "fold": fold,
                "weight_mean": float(weights.mean()) if len(weights) else None,
                "weight_min": float(weights.min()) if len(weights) else None,
                "weight_max": float(weights.max()) if len(weights) else None,
                "weight_p95": float(np.quantile(weights, 0.95)) if len(weights) else None,
                "release_gate_weight_mean": float(weights[release_mask].mean()) if release_mask.any() else None,
                "release_gate_weight_max": float(weights[release_mask].max()) if release_mask.any() else None,
                "release_gate_weight_share": release_weight_sum / weight_sum if weight_sum > 0 else None,
                "release_gate_samples": int(release_mask.sum()),
            }
        )

        bc_result = fit_state_objective_bc_actor_from_frame(
            frame,
            targets=fold_targets,
            profile=profile,
            train_config=train_cfg,
            sample_weight_config=weight_cfg,
            sample_size=0,
            rng_seed=SEED,
        )
        slice_metrics = evaluate_bc_actor_slice_fit(
            bc_result.actor,
            bc_result.obs,
            bc_result.actions,
            dataset.metadata,
            slice_cols=SLICE_COLS,
        )
        slice_gate, slice_details = slice_gate_from_metrics(slice_metrics)
        tagged_slices = slice_metrics.copy()
        tagged_slices.insert(0, "fold", fold)
        tagged_slices.insert(0, "config", config_name)
        slice_frames.append(tagged_slices)

        trace = release_gate_trace(bc_result.actor, bc_result.obs, bc_result.actions, dataset.metadata)
        if not trace.empty:
            trace.insert(0, "fold", fold)
            trace.insert(0, "config", config_name)
            trace_frames.append(trace)

        action_distribution = bc_result.action_distribution.copy()
        action_distribution.insert(0, "fold", fold)
        action_distribution.insert(0, "config", config_name)
        if action_levels:
            action_distribution["action_level"] = action_distribution["action"].map(
                {idx: level for idx, level in enumerate(action_levels)}
            )
        action_frames.append(action_distribution)

        before_loss = float(bc_result.metrics["before_loss"])
        after_loss = float(bc_result.metrics["after_loss"])
        before_accuracy = float(bc_result.metrics["before_accuracy"])
        after_accuracy = float(bc_result.metrics["after_accuracy"])
        all_gate = bool(after_loss < before_loss and after_accuracy >= SLICE_RULES["all"]["min_accuracy"])
        fit_gate = bool(all_gate and slice_gate)
        dataset_row = dataset_summary[dataset_summary["fold"].eq(fold)].iloc[0].to_dict()
        fold_rows.append(
            {
                "config": config_name,
                "fold": fold,
                "status": "evaluated",
                "fit_gate": fit_gate,
                "all_gate": all_gate,
                "slice_gate": bool(slice_gate),
                "failed_required_slices": ",".join(slice_details["failed_required_slices"]),
                "before_loss": before_loss,
                "after_loss": after_loss,
                "loss_improvement": before_loss - after_loss,
                "before_accuracy": before_accuracy,
                "after_accuracy": after_accuracy,
                "train_last_loss": float(bc_result.metrics["train_last_loss"]),
                "train_last_batch_accuracy": float(bc_result.metrics["train_last_batch_accuracy"]),
                "samples": int(bc_result.metrics["samples"]),
                "steps": int(bc_result.metrics["steps"]),
                "lr": float(bc_result.metrics["lr"]),
                "batch_size": int(bc_result.metrics["batch_size"]),
                "weight_mean": float(bc_result.metrics.get("weight_mean", float("nan"))),
                "weight_min": float(bc_result.metrics.get("weight_min", float("nan"))),
                "weight_max": float(bc_result.metrics.get("weight_max", float("nan"))),
                **dataset_row,
            }
        )

fold_config_metrics = pd.DataFrame(fold_rows)
state_slice_config_metrics = pd.concat(slice_frames, ignore_index=True) if slice_frames else pd.DataFrame()
sample_weight_config_metrics = pd.DataFrame(weight_rows)
release_gate_prediction_trace = pd.concat(trace_frames, ignore_index=True) if trace_frames else pd.DataFrame()
bc_action_distribution = pd.concat(action_frames, ignore_index=True) if action_frames else pd.DataFrame()

fold_config_metrics.to_csv(CARD / "fold_config_metrics.csv", index=False)
state_slice_config_metrics.to_csv(CARD / "state_slice_config_metrics.csv", index=False)
sample_weight_config_metrics.to_csv(CARD / "sample_weight_config_metrics.csv", index=False)
release_gate_prediction_trace.to_csv(CARD / "release_gate_prediction_trace.csv", index=False)
bc_action_distribution.to_csv(CARD / "bc_action_distribution.csv", index=False)

display(fold_config_metrics)
display(state_slice_config_metrics[state_slice_config_metrics["slice"].eq("release_gate_state")])
display(sample_weight_config_metrics)


config current_readiness_baseline
  fold wf_2020


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


  fold wf_2022


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


  fold wf_2023_2024


...

  fold wf_2023_2024


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


,config,fold,status,fit_gate,all_gate,slice_gate,failed_required_slices,before_loss,after_loss,loss_improvement,...,valid_overlap_rows,test_overlap_rows,target_min_date,target_max_date,target_scope_ok,bc_samples,metadata_rows,unique_label_actions,release_gate_samples,feature_coverage_ok
0,current_readiness_baseline,wf_2020,evaluated,True,True,True,,1.605351,1.183302,0.422050,...,252,0,2010-03-11,2018-12-31,True,1905,1905,5,32,True
1,current_readiness_baseline,wf_2022,evaluated,True,True,True,,1.614201,1.032363,0.581837,...,252,0,2020-01-02,2020-12-31,True,253,253,5,8,True
2,current_readiness_baseline,wf_2023_2024,evaluated,False,True,False,release_gate_state,1.613803,1.167221,0.446582,...,0,0,2020-01-02,2022-12-30,True,756,756,5,11,True
3,release64_cb99_obj020,wf_2020,evaluated,True,True,True,,1.610265,1.239112,0.371153,...,252,0,2010-03-11,2018-12-31,True,1905,1905,5,32,True
4,release64_cb99_obj020,wf_2022,evaluated,True,True,True,,1.620834,1.121073,0.499762,...,252,0,2020-01-02,2020-12-31,True,253,253,5,8,True
5,release64_cb99_obj020,wf_2023_2024,evaluated,True,True,True,,1.614710,1.184070,0.430640,...,0,0,2020-01-02,2022-12-30,True,756,756,5,11,True
6,release128_cb995_obj020,wf_2020,evaluated,True,True,True,,1.611309,1.296018,0.315292,...,252,0,2010-03-11,2018-12-31,True,1905,1905,5,32,True
7,release128_cb995_obj020,wf_2022,evaluated,True,True,True,,1.614189,1.100266,0.513924,...,252,0,2020-01-02,2020-12-31,True,253,253,5,8,True
8,release128_cb995_obj020,wf_2023_2024,evaluated,True,True,True,,1.613835,1.170003,0.443833,...,0,0,2020-01-02,2022-12-30,True,756,756,5,11,True
9,release128_cb995_obj000,wf_2020,evaluated,True,True,True,,1.603991,1.277997,0.325994,...,252,0,2010-03-11,2018-12-31,True,1905,1905,5,32,True


,config,fold,slice,samples,accuracy,loss,label_unique_actions,predicted_unique_actions,label_dominant_action,label_dominant_share,predicted_dominant_action,predicted_dominant_share
4,current_readiness_baseline,wf_2020,release_gate_state,32,0.750000,1.147921,3,3,4,0.843750,4,0.718750
9,current_readiness_baseline,wf_2022,release_gate_state,8,0.875000,1.023031,4,3,4,0.375000,4,0.375000
14,current_readiness_baseline,wf_2023_2024,release_gate_state,11,0.363636,1.557956,4,1,3,0.454545,4,1.000000
19,release64_cb99_obj020,wf_2020,release_gate_state,32,1.000000,0.928645,3,3,4,0.843750,4,0.843750
24,release64_cb99_obj020,wf_2022,release_gate_state,8,1.000000,0.922206,4,4,4,0.375000,4,0.375000
29,release64_cb99_obj020,wf_2023_2024,release_gate_state,11,0.727273,1.178338,4,2,3,0.454545,4,0.545455
34,release128_cb995_obj020,wf_2020,release_gate_state,32,0.968750,0.939497,3,3,4,0.843750,4,0.812500
39,release128_cb995_obj020,wf_2022,release_gate_state,8,1.000000,0.904943,4,4,4,0.375000,4,0.375000
44,release128_cb995_obj020,wf_2023_2024,release_gate_state,11,0.818182,1.092558,4,2,3,0.454545,3,0.545455
49,release128_cb995_obj000,wf_2020,release_gate_state,32,1.000000,0.909967,3,3,4,0.843750,4,0.843750


,config,fold,weight_mean,weight_min,weight_max,weight_p95,release_gate_weight_mean,release_gate_weight_max,release_gate_weight_share,release_gate_samples
0,current_readiness_baseline,wf_2020,1.0,0.394806,14.723673,1.486623,11.350637,14.723673,0.190667,32
1,current_readiness_baseline,wf_2022,1.0,0.351684,6.909615,4.702914,6.566516,6.909615,0.207637,8
2,current_readiness_baseline,wf_2023_2024,1.0,0.226098,9.619936,2.185272,6.612840,9.619936,0.096219,11
3,release64_cb99_obj020,wf_2020,1.0,0.287204,24.481920,1.081454,24.481920,24.481920,0.411245,32
4,release64_cb99_obj020,wf_2022,1.0,0.272252,12.226275,3.640705,12.226275,12.226275,0.386602,8
5,release64_cb99_obj020,wf_2023_2024,1.0,0.184062,17.900362,1.778988,15.333384,17.900362,0.223105,11
6,release128_cb995_obj020,wf_2020,1.0,0.201029,32.980778,0.956476,32.980778,32.980778,0.554008,32
7,release128_cb995_obj020,wf_2022,1.0,0.190852,16.683311,3.178212,16.683311,16.683311,0.527535,8
8,release128_cb995_obj020,wf_2023_2024,1.0,0.137463,26.215343,1.414903,22.973873,26.215343,0.334276,11
9,release128_cb995_obj000,wf_2020,1.0,0.265733,43.596149,0.883550,34.945297,43.596149,0.587008,32


In [6]:

required_folds = set(WALK_FORWARD_WINDOWS)
evaluated = fold_config_metrics[fold_config_metrics["status"].eq("evaluated")].copy()
release_gate_metrics = state_slice_config_metrics[state_slice_config_metrics["slice"].eq("release_gate_state")].copy()
release_gate_metrics["release_gate_pass"] = release_gate_metrics["accuracy"] >= SLICE_RULES["release_gate_state"]["min_accuracy"]

summary_rows: list[dict[str, Any]] = []
for config_name, group in evaluated.groupby("config"):
    release_group = release_gate_metrics[release_gate_metrics["config"].eq(config_name)].copy()
    scored_folds = set(group["fold"].astype(str).tolist())
    all_required_evaluated = scored_folds == required_folds
    fit_gate_count = int(group["fit_gate"].fillna(False).astype(bool).sum())
    release_gate_pass_count = int(release_group["release_gate_pass"].fillna(False).astype(bool).sum())
    wf2023_release = release_group[release_group["fold"].eq("wf_2023_2024")]
    wf2023_accuracy = float(wf2023_release["accuracy"].iloc[0]) if not wf2023_release.empty else float("nan")
    wf2023_pred_unique = int(wf2023_release["predicted_unique_actions"].iloc[0]) if not wf2023_release.empty else 0
    summary_rows.append(
        {
            "config": config_name,
            "evaluated_fold_count": int(len(group)),
            "all_required_folds_evaluated": bool(all_required_evaluated),
            "fit_gate_count": fit_gate_count,
            "slice_gate_count": int(group["slice_gate"].fillna(False).astype(bool).sum()),
            "all_gate_count": int(group["all_gate"].fillna(False).astype(bool).sum()),
            "release_gate_pass_count": release_gate_pass_count,
            "all_required_fit_gate": bool(all_required_evaluated and fit_gate_count == len(required_folds)),
            "release_gate_accuracy_min": float(release_group["accuracy"].min()) if not release_group.empty else float("nan"),
            "wf_2023_2024_release_gate_accuracy": wf2023_accuracy,
            "wf_2023_2024_release_gate_predicted_unique_actions": wf2023_pred_unique,
            "all_accuracy_min": float(group["after_accuracy"].min()) if not group.empty else float("nan"),
            "after_accuracy_mean": float(group["after_accuracy"].mean()) if not group.empty else float("nan"),
            "after_loss_mean": float(group["after_loss"].mean()) if not group.empty else float("nan"),
            "failed_folds": sorted(group.loc[~group["fit_gate"].fillna(False).astype(bool), "fold"].astype(str).tolist()),
            "failed_required_slices": sorted(
                s for s in group.get("failed_required_slices", pd.Series(dtype=str)).fillna("").astype(str).unique().tolist() if s
            ),
        }
    )

candidate_summary = pd.DataFrame(summary_rows)
ranked_candidates = candidate_sort_key(candidate_summary)
selected_row = ranked_candidates.iloc[0].to_dict() if not ranked_candidates.empty else {}
selected_config = str(selected_row.get("config", ""))
all_target_scope_ok = bool(scope_audit["target_scope_gate"] and dataset_summary["target_scope_ok"].fillna(False).astype(bool).all())
all_feature_coverage_ok = bool(dataset_summary.get("feature_coverage_ok", pd.Series(False, index=dataset_summary.index)).fillna(False).astype(bool).all())
all_fit_gates = bool(selected_row.get("all_required_fit_gate", False))
selected_ready = bool(all_fit_gates and all_target_scope_ok and all_feature_coverage_ok)

decision = (
    "bc_repair_weighting_ready_for_checkpoint_replay_no_promotion"
    if selected_ready
    else "bc_repair_weighting_failed_no_ppo"
)
next_mechanism = (
    "wire_repaired_bc_weighting_into_checkpoint_metric_replay_before_ppo"
    if selected_ready
    else "add_release_gate_auxiliary_state_head_or_target_features_before_ppo"
)

candidate_summary.to_csv(CARD / "candidate_summary.csv", index=False)
ranked_candidates.to_csv(CARD / "candidate_ranking.csv", index=False)

selection = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "next_mechanism": next_mechanism,
    "selected_config": selected_config,
    "selected_ready": selected_ready,
    "all_target_scope_ok": all_target_scope_ok,
    "all_feature_coverage_ok": all_feature_coverage_ok,
    "selected_summary": selected_row,
    "candidate_summary": candidate_summary.to_dict("records"),
    "promotion_allowed": False,
    "actor_retention_allowed": False,
}
write_json(CARD / "selection.json", selection)

display(candidate_summary)
display(ranked_candidates.head(10))
print("decision", decision, "selected_config", selected_config, "next", next_mechanism)


,config,evaluated_fold_count,all_required_folds_evaluated,fit_gate_count,slice_gate_count,all_gate_count,release_gate_pass_count,all_required_fit_gate,release_gate_accuracy_min,wf_2023_2024_release_gate_accuracy,wf_2023_2024_release_gate_predicted_unique_actions,all_accuracy_min,after_accuracy_mean,after_loss_mean,failed_folds,failed_required_slices
0,current_readiness_baseline,3,True,2,2,3,2,False,0.363636,0.363636,1,0.724934,0.780172,1.127629,[wf_2023_2024],[release_gate_state]
1,release128_cb995_obj000,3,True,3,3,3,3,True,0.818182,0.818182,2,0.623622,0.696293,1.208503,[],[]
2,release128_cb995_obj020,3,True,3,3,3,3,True,0.818182,0.818182,2,0.601050,0.714721,1.188762,[],[]
3,release256_beta0_obj020,3,True,3,3,3,3,True,0.818182,0.818182,2,0.558530,0.677709,1.224358,[],[]
4,release256_cb995_obj020,3,True,3,3,3,3,True,0.818182,0.818182,2,0.592651,0.714120,1.189905,[],[]
5,release64_cb99_obj020,3,True,3,3,3,3,True,0.727273,0.727273,2,0.668766,0.723210,1.181418,[],[]


,config,evaluated_fold_count,all_required_folds_evaluated,fit_gate_count,slice_gate_count,all_gate_count,release_gate_pass_count,all_required_fit_gate,release_gate_accuracy_min,wf_2023_2024_release_gate_accuracy,wf_2023_2024_release_gate_predicted_unique_actions,all_accuracy_min,after_accuracy_mean,after_loss_mean,failed_folds,failed_required_slices
1,release128_cb995_obj000,3,True,3,3,3,3,True,0.818182,0.818182,2,0.623622,0.696293,1.208503,[],[]
2,release128_cb995_obj020,3,True,3,3,3,3,True,0.818182,0.818182,2,0.601050,0.714721,1.188762,[],[]
4,release256_cb995_obj020,3,True,3,3,3,3,True,0.818182,0.818182,2,0.592651,0.714120,1.189905,[],[]
3,release256_beta0_obj020,3,True,3,3,3,3,True,0.818182,0.818182,2,0.558530,0.677709,1.224358,[],[]
5,release64_cb99_obj020,3,True,3,3,3,3,True,0.727273,0.727273,2,0.668766,0.723210,1.181418,[],[]
0,current_readiness_baseline,3,True,2,2,3,2,False,0.363636,0.363636,1,0.724934,0.780172,1.127629,[wf_2023_2024],[release_gate_state]


decision bc_repair_weighting_ready_for_checkpoint_replay_no_promotion selected_config release128_cb995_obj000 next wire_repaired_bc_weighting_into_checkpoint_metric_replay_before_ppo


In [7]:

retention_summary = retention_gate.get("summary", {}) if isinstance(retention_gate, dict) else {}
source_failed_slice = {
    "fold": "wf_2023_2024",
    "slice": "release_gate_state",
    "accuracy": 0.36363636363636365,
    "samples": 11,
    "predicted_unique_actions": 1,
    "predicted_dominant_action": 4,
}

output_paths = {
    "config_grid": CARD / "config_grid.csv",
    "dataset_summary": CARD / "dataset_summary.csv",
    "fold_config_metrics": CARD / "fold_config_metrics.csv",
    "state_slice_config_metrics": CARD / "state_slice_config_metrics.csv",
    "sample_weight_config_metrics": CARD / "sample_weight_config_metrics.csv",
    "release_gate_prediction_trace": CARD / "release_gate_prediction_trace.csv",
    "bc_action_distribution": CARD / "bc_action_distribution.csv",
    "candidate_summary": CARD / "candidate_summary.csv",
    "candidate_ranking": CARD / "candidate_ranking.csv",
    "feature_schema": CARD / "feature_schema.csv",
    "feature_summary": CARD / "feature_summary.json",
    "target_scope_audit": CARD / "target_scope_audit.csv",
    "selection": CARD / "selection.json",
    "summary": CARD / "summary.json",
    "conclusion": CARD / "conclusion.md",
}

summary = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "next_mechanism": next_mechanism,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "role": "diagnose_bc_weight_repair",
    "canonical_baseline": "soxlbacktest_policy_reimplementation_v1",
    "direct_comparison_baseline": "release16_checkpoint_distillation_bc_readiness_v1",
    "target_window": list(WALK_FORWARD_WINDOWS),
    "changed_layer": ["model_selection", "bc_sample_weighting", "checkpoint_distillation_handoff"],
    "external_reference": "converted SOXLBacktest qlib buy/sell baseline plus release16 state-objective contract",
    "hypothesis": "The fold-local checkpoint-distillation target contract can proceed only if a BC weighting repair makes all required sparse state slices pass without losing global fit.",
    "acceptance_criteria": [
        "target_scope_gate passes",
        "all required walk-forward folds have train-local labels",
        "feature frame covers all labels",
        "selected BC weighting config passes all required slice gates",
        "no actor retained or promoted from this screen",
    ],
    "expected_tradeoff": "Allow a slightly heavier release-gate supervised warm start only if it keeps global fit and required fold gates intact.",
    "failure_mode": "Overweighting sparse release-gate rows may overfit 11 samples, collapse predictions to a different action, or damage ordinary/downtrend fit before PPO.",
    "comparison_table_row_key": EXPERIMENT,
    "closest_historical_success_case": source_contract_summary.get("closest_historical_success_case", "release16 wf_2022 BC initialization passed a checkpoint replay gate"),
    "closest_historical_failure_case": source_readiness_summary.get("closest_historical_failure_case", "release16 BC readiness failed wf_2023_2024 release_gate_state"),
    "why_not_duplicate": "This screen changes only BC sample weighting around the known failed sparse slice; it does not rebuild the target contract, rerun PPO, or select an actor checkpoint.",
    "profile": profile.name,
    "feature_summary": feature_summary,
    "source_contract_decision": source_contract_summary.get("decision"),
    "source_readiness_decision": source_readiness_summary.get("decision"),
    "source_failed_slice": source_failed_slice,
    "target_scope_audit": scope_audit,
    "dataset_summary": dataset_summary.to_dict("records"),
    "config_count": int(len(CONFIG_GRID)),
    "candidate_summary": candidate_summary.to_dict("records"),
    "selected_config": selected_config,
    "selected_summary": selected_row,
    "gate_summary": {
        "all_target_scope_ok": all_target_scope_ok,
        "all_feature_coverage_ok": all_feature_coverage_ok,
        "selected_all_fit_gates": all_fit_gates,
        "selected_ready": selected_ready,
        "required_fold_count": len(WALK_FORWARD_WINDOWS),
        "best_release_gate_pass_count": int(selected_row.get("release_gate_pass_count", 0)) if selected_row else 0,
    },
    "retention_gate_current": {
        "passing_profile_count": retention_summary.get("passing_profile_count"),
        "max_walkforward_drawdown_pct": retention_summary.get("max_walkforward_drawdown_pct"),
        "current_best_return_pct": retention_summary.get("current_best_return_pct"),
        "required_best_return_pct": retention_summary.get("required_best_return_pct"),
        "reference_profile": retention_summary.get("reference_profile"),
        "reference_return_pct": retention_summary.get("reference_return_pct"),
    },
    "output_paths": {key: str(path.relative_to(ROOT)) for key, path in output_paths.items()},
    "conclusion": (
        "Selected BC weighting is ready for checkpoint replay; still no promotion until official daily replay and PPO drift gates pass."
        if selected_ready
        else "Do not launch PPO; weighting-only repair did not make every required BC slice pass, so representation/target features need repair first."
    ),
}
write_json(CARD / "summary.json", summary)

conclusion_lines = [
    f"# {EXPERIMENT}",
    "",
    f"Decision: `{decision}`.",
    f"Selected config: `{selected_config}`.",
    f"Next mechanism: `{next_mechanism}`.",
    "",
    "No actor checkpoint is retained or promoted from this screen.",
    "",
    "Key evidence:",
    f"- Source failure: `wf_2023_2024 release_gate_state` accuracy {source_failed_slice['accuracy']:.4f} on {source_failed_slice['samples']} samples with one predicted action.",
    f"- Best release-gate pass count: {int(selected_row.get('release_gate_pass_count', 0)) if selected_row else 0} / {len(WALK_FORWARD_WINDOWS)}.",
    f"- Selected `wf_2023_2024 release_gate_state` accuracy: {selected_row.get('wf_2023_2024_release_gate_accuracy') if selected_row else None}.",
    f"- Selected all-fit gate: {bool(selected_row.get('all_required_fit_gate', False)) if selected_row else False}.",
]
(CARD / "conclusion.md").write_text("\n".join(conclusion_lines) + "\n", encoding="utf-8")

display(Markdown("\n".join(conclusion_lines)))


# release16_checkpoint_distillation_bc_repair_screen_v1

Decision: `bc_repair_weighting_ready_for_checkpoint_replay_no_promotion`.
Selected config: `release128_cb995_obj000`.
Next mechanism: `wire_repaired_bc_weighting_into_checkpoint_metric_replay_before_ppo`.

No actor checkpoint is retained or promoted from this screen.

Key evidence:
- Source failure: `wf_2023_2024 release_gate_state` accuracy 0.3636 on 11 samples with one predicted action.
- Best release-gate pass count: 3 / 3.
- Selected `wf_2023_2024 release_gate_state` accuracy: 0.8181818181818182.
- Selected all-fit gate: True.